# 03 — Modelagem, Avaliação e InterpretaçãoCobre as **Etapas 4, 5 e 6** do enunciado:4. Treinar **no mínimo dois** modelos e comparar.5. Avaliar com métricas adequadas.6. Identificar variáveis influentes e discutir implicações para a produção.> Responsável: _(Modelagem & Avaliação)_

In [ ]:
import syssys.path.append("..")import matplotlib.pyplot as pltimport pandas as pdimport seaborn as snsfrom src import modeling as md_from src import evaluation as evpd.set_option("display.max_columns", None)sns.set_theme(style="whitegrid")

In [ ]:
X_train = pd.read_csv("../data/processed/X_train.csv")X_test  = pd.read_csv("../data/processed/X_test.csv")y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()y_test  = pd.read_csv("../data/processed/y_test.csv").squeeze()print("Treino:", X_train.shape, "| Teste:", X_test.shape)

## Etapa 4 — Desenvolvimento dos modelosDupla principal:- **Regressão Logística** — baseline linear e interpretável; os coeficientes dizem a direção do efeito.- **Random Forest** — captura relações não-lineares e entrega importância de variáveis para a Etapa 6.Os demais (Gradient Boosting, KNN, SVM) enriquecem a comparação e cobrem o conteúdo da fase.`class_weight="balanced"` compensa o desbalanceamento visto na EDA.

In [ ]:
modelos = md_.get_models()fitted = md_.train_models(modelos, X_train, y_train)list(fitted)

### Validação cruzada estratificada (5 folds)

In [ ]:
cv = md_.cross_validate_models(modelos, X_train, y_train, scoring="f1", cv=5)display(cv)

## Etapa 5 — Avaliação e comparação> ⚠️ **Não decidam por acurácia.** Com classes desbalanceadas, ela premia o modelo que ignora> a classe minoritária. O critério de escolha aqui é **F1 e ROC-AUC**, olhando o **Recall** da> classe "alta qualidade" — ou seja, quantos vinhos bons o modelo realmente encontra.

In [ ]:
comparativo = ev.compare_models(fitted, X_test, y_test)display(comparativo)

In [ ]:
for nome, m in fitted.items():    ev.print_report(m, X_test, y_test, nome)

In [ ]:
ev.plot_roc(fitted, X_test, y_test)plt.show()

In [ ]:
MELHOR = comparativo.iloc[0]["modelo"]print("Melhor modelo por F1:", MELHOR)ev.plot_confusion(fitted[MELHOR], X_test, y_test, MELHOR)plt.show()

**Leitura da matriz de confusão em linguagem de negócio:**- **Falso negativo** = vinho bom classificado como comum → perda de valor comercial.- **Falso positivo** = vinho comum classificado como bom → risco de reputação da marca._Qual dos dois erros custa mais caro para a vinícola? A resposta define se otimizamos Recall ouPrecision — e essa discussão é exatamente o tipo de coisa que rende ponto na apresentação._

### Ajuste de hiperparâmetros (opcional, se sobrar tempo)

In [ ]:
# best_rf, best_params, best_score = md_.tune_random_forest(X_train, y_train)# print(best_params, best_score)# fitted["Random Forest (tunada)"] = best_rf

## Etapa 6 — Interpretação dos resultados

In [ ]:
imp = md_.feature_importance(fitted["Random Forest"], X_train.columns)display(imp)ev.plot_feature_importance(imp, top=10, name="Random Forest")plt.show()

In [ ]:
# Direção do efeito: o coeficiente da logística diz se a variável ajuda ou atrapalhacoef = pd.DataFrame({    "variavel": X_train.columns,    "coeficiente": fitted["Regressao Logistica"].coef_[0],}).sort_values("coeficiente", ascending=False)coef["efeito"] = coef["coeficiente"].apply(lambda v: "aumenta a chance" if v > 0 else "reduz a chance")display(coef)

### Conclusões (preencher)**Variáveis mais influentes na qualidade:**1. _..._2. _..._3. _..._**Implicações para o processo produtivo:**| Variável | Efeito | O que o produtor pode fazer ||---|---|---|| _ex.: acidez volátil_ | reduz a qualidade | controle microbiológico e higiene na fermentação para evitar acetificação || _ex.: teor alcoólico_ | aumenta a qualidade | monitorar o ponto de colheita / maturação da uva || _..._ | | |**Recomendação final para a diretoria:** _uma frase, sem jargão técnico. Esta é a frase queabre o vídeo executivo._

In [ ]:
# Exportar tabelas finais para results/comparativo.to_csv("../results/metrics/comparativo_modelos.csv", index=False)imp.to_csv("../results/metrics/importancia_variaveis.csv", index=False)coef.to_csv("../results/metrics/coeficientes_logistica.csv", index=False)print("Métricas exportadas para results/metrics/")